# Seleccion de Caracteristicas por Entropia

## Datos del estudiante
- **Nombre:** Angel David Pineros Sierra
- **Asignatura:** Mineria de Datos
- **Tema:** Seleccion de caracteristicas basada en entropia
- **Fecha:** 11 De marzo de 2026

---

## Descripcion de la tarea
En este notebook se construye un procedimiento de seleccion de caracteristicas basado en entropia. La idea central es medir cuanta informacion se pierde cuando se elimina una variable del conjunto de datos y, a partir de ello, construir un ranking de variables desde la menos relevante hasta la mas relevante.


## Metodologia general

El procedimiento implementado en este notebook sigue estas ideas:

1. Se parte del conjunto completo de variables del `DataFrame` `df`.
2. Se calcula la similitud entre pares de registros a partir de coincidencias exactas entre columnas.
3. Con esa similitud se calcula la entropia del conjunto.
4. Luego se elimina una variable a la vez de manera temporal y se recalcula la entropia.
5. La variable cuya eliminacion produce la menor diferencia de entropia se considera la menos relevante en esa iteracion.
6. El proceso se repite hasta dejar una sola variable, obteniendo asi un ranking de eliminacion y una variable final seleccionada.


In [1]:
import pandas as pd

data = {
    'F1': ['A', 'B', 'C', 'B', 'C'],
    'F2': ['X', 'Y', 'Y', 'X', 'Z'],
    'F3': [1, 2, 2, 1, 3]
}

df = pd.DataFrame(data)
display(df)

,F1,F2,F3
0,A,X,1
1,B,Y,2
2,C,Y,2
3,B,X,1
4,C,Z,3


## Construccion del conjunto de datos y de la matriz de similitud

En las siguientes celdas se crea un `DataFrame` de ejemplo y se calcula una matriz de similitud tipo Hamming normalizada. Para cada par de filas `x_i` y `x_j`, la similitud `S_ij` corresponde a la proporcion de columnas en las que ambos registros tienen exactamente el mismo valor.

Esta matriz es la base para el calculo posterior de la entropia del conjunto.


In [2]:
import numpy as np

num_rows = df.shape[0]
hamming_similarity_matrix = np.zeros((num_rows, num_rows))

for i in range(num_rows):
    for j in range(num_rows):
        row1 = df.iloc[i]
        row2 = df.iloc[j]

        matching_features = 0
        total_features = df.shape[1]

        for col in df.columns:
            if row1[col] == row2[col]:
                matching_features += 1

        hamming_similarity_matrix[i, j] = matching_features / total_features

# Convert to DataFrame for better display with row/column labels
hamming_similarity_df = pd.DataFrame(hamming_similarity_matrix,
                                     index=[f'R{i+1}' for i in range(num_rows)],
                                     columns=[f'R{i+1}' for i in range(num_rows)])

# Create a copy to modify for the upper triangular form
upper_triangular_hamming_df = hamming_similarity_df.copy()

# Set diagonal to 0
np.fill_diagonal(upper_triangular_hamming_df.values, 0)

# Set lower triangular part to 0
for i in range(num_rows):
    for j in range(i): # Corrected line: iterate j from 0 to i-1
        upper_triangular_hamming_df.iloc[i, j] = 0

display(upper_triangular_hamming_df)

,R1,R2,R3,R4,R5
R1,0.0,0.0,0.000000,0.666667,0.000000
R2,0.0,0.0,0.666667,0.333333,0.000000
R3,0.0,0.0,0.000000,0.000000,0.333333
R4,0.0,0.0,0.000000,0.000000,0.000000
R5,0.0,0.0,0.000000,0.000000,0.000000


## Entropia del conjunto completo

A partir de la matriz de similitud se calcula la entropia total del conjunto. Esta medida resume el grado de incertidumbre o dispersion existente entre los registros del dataset.



En todos los casos, cuando `S_ij` es `0` o `1`, la contribucion de ese par a la entropia se toma como `0`.


In [3]:
# Calculo de la entropia del conjunto usando la similitud S_ij
# Cambia log_base por 2, np.e, 10 o cualquier otra base valida.

log_base = 10

def entropy_term(s, base=log_base):
    if s == 0 or s == 1:
        return 0.0
    return -(s * (np.log(s) / np.log(base)) + (1 - s) * (np.log(1 - s) / np.log(base)))

E = 0.0
entropy_details = []

for i in range(num_rows - 1):
    for j in range(i + 1, num_rows):
        s_ij = hamming_similarity_matrix[i, j]
        term = entropy_term(s_ij, base=log_base)
        E += term
        entropy_details.append([f'R{i+1}', f'R{j+1}', s_ij, term])

entropy_details_df = pd.DataFrame(
    entropy_details,
    columns=['Fila_i', 'Fila_j', 'S_ij', 'Contribucion']
)

display(entropy_details_df)
print('Base del logaritmo =', log_base)
print('Entropia total E =', E)


,Fila_i,Fila_j,S_ij,Contribucion
0,R1,R2,0.000000,0.000000
1,R1,R3,0.000000,0.000000
2,R1,R4,0.666667,0.276435
3,R1,R5,0.000000,0.000000
4,R2,R3,0.666667,0.276435
5,R2,R4,0.333333,0.276435
6,R2,R5,0.000000,0.000000
7,R3,R4,0.000000,0.000000
8,R3,R5,0.333333,0.276435
9,R4,R5,0.000000,0.000000


Base del logaritmo = 10
Entropia total E = 1.1057383637746998


## Algoritmo de seleccion de caracteristicas

El algoritmo implementado a continuacion es generico y funciona con cualquier `DataFrame` llamado `df`.

### Logica de cada iteracion
En cada iteracion se realiza el siguiente proceso:

1. Se calcula la entropia del conjunto actual de variables.
2. Se elimina temporalmente una variable candidata.
3. Se recalcula la entropia del subconjunto resultante.
4. Se mide la diferencia `delta_entropia = E(F) - E(F - f)`.
5. La variable con menor `delta_entropia` es la que menos afecta la informacion total del conjunto y, por tanto, se elimina en esa iteracion.

Si varias variables producen exactamente la misma menor diferencia de entropia, el notebook reporta el empate de forma explicita. Ademas, si en una iteracion **todos** los `delta_entropia` son iguales, el algoritmo se detiene y no elimina ninguna variable, ya que con este criterio no existe evidencia para retirar una sin afectar de forma equivalente la variabilidad del sistema.


In [4]:
# Funciones genericas para seleccion de caracteristicas por entropia.

def entropy_term(s, base=2):
    if s == 0 or s == 1:
        return 0.0
    return -(s * (np.log(s) / np.log(base)) + (1 - s) * (np.log(1 - s) / np.log(base)))

def similarity_matrix_from_columns(dataframe, columns):
    num_rows = dataframe.shape[0]
    matrix = np.zeros((num_rows, num_rows))

    for i in range(num_rows):
        for j in range(num_rows):
            row1 = dataframe.iloc[i][columns]
            row2 = dataframe.iloc[j][columns]
            matches = (row1 == row2).sum()
            matrix[i, j] = matches / len(columns)

    return matrix

def dataset_entropy(dataframe, columns=None, base=2):
    if columns is None:
        columns = list(dataframe.columns)

    if len(columns) == 0:
        raise ValueError('Se necesita al menos una columna para calcular la entropia.')

    similarity_matrix = similarity_matrix_from_columns(dataframe, columns)
    total_entropy = 0.0
    num_rows = similarity_matrix.shape[0]

    for i in range(num_rows - 1):
        for j in range(i + 1, num_rows):
            total_entropy += entropy_term(similarity_matrix[i, j], base=base)

    return total_entropy

def entropy_feature_ranking(dataframe, base=2, atol=1e-12):
    current_columns = list(dataframe.columns)

    if len(current_columns) < 2:
        raise ValueError('El algoritmo necesita al menos dos columnas en df.')

    ranking = []
    iteration_tables = []
    iteration = 1

    while len(current_columns) > 1:
        current_entropy = dataset_entropy(dataframe, current_columns, base=base)
        candidate_rows = []

        for column in current_columns:
            reduced_columns = [col for col in current_columns if col != column]
            reduced_entropy = dataset_entropy(dataframe, reduced_columns, base=base)
            delta_entropy = current_entropy - reduced_entropy

            candidate_rows.append({
                'iteracion': iteration,
                'columna_removida_candidata': column,
                'entropia_actual': current_entropy,
                'entropia_sin_columna': reduced_entropy,
                'delta_entropia': delta_entropy
            })

        min_delta = min(row['delta_entropia'] for row in candidate_rows)
        tied_columns = [
            row['columna_removida_candidata']
            for row in candidate_rows
            if np.isclose(row['delta_entropia'], min_delta, atol=atol, rtol=0)
        ]

        for row in candidate_rows:
            row['empate_minimo'] = row['columna_removida_candidata'] in tied_columns

        all_equal_deltas = all(
            np.isclose(row['delta_entropia'], candidate_rows[0]['delta_entropia'], atol=atol, rtol=0)
            for row in candidate_rows
        )

        if all_equal_deltas:
            iteration_tables.append({
                'iteracion': iteration,
                'tabla': pd.DataFrame(candidate_rows),
                'entropia_actual': current_entropy,
                'empatadas': tied_columns,
                'columna_eliminada': None,
                'columnas_restantes': current_columns.copy(),
                'detener_por_deltas_iguales': True,
                'mensaje': 'No se elimina ninguna variable porque todos los deltas son iguales y retirar una variable puede eliminar variabilidad del sistema.'
            })

            return {
                'ranking_eliminacion': ranking,
                'columna_final': None,
                'columnas_restantes': current_columns,
                'iteraciones': iteration_tables,
                'detenido_por_deltas_iguales': True
            }

        selected_column = tied_columns[0]
        current_columns.remove(selected_column)
        ranking.append(selected_column)

        iteration_tables.append({
            'iteracion': iteration,
            'tabla': pd.DataFrame(candidate_rows),
            'entropia_actual': current_entropy,
            'empatadas': tied_columns,
            'columna_eliminada': selected_column,
            'columnas_restantes': current_columns.copy(),
            'detener_por_deltas_iguales': False,
            'mensaje': ''
        })

        iteration += 1

    return {
        'ranking_eliminacion': ranking,
        'columna_final': current_columns[0],
        'columnas_restantes': current_columns,
        'iteraciones': iteration_tables,
        'detenido_por_deltas_iguales': False
    }


## Interpretacion del ranking y seleccion final

La siguiente celda ejecuta el algoritmo y presenta una tabla por iteracion.

### Como leer cada iteracion
- **`entropia_actual`**: entropia del conjunto antes de eliminar una variable.
- **`entropia_sin_columna`**: entropia del conjunto al retirar temporalmente una variable candidata.
- **`delta_entropia`**: perdida de informacion asociada con la eliminacion de esa variable.
- **`empate_minimo`**: indica si la variable pertenece al grupo con la menor diferencia de entropia.

### Criterio de ranking
- Las primeras variables eliminadas son las menos relevantes segun el criterio de entropia.
- Las variables que sobreviven mas iteraciones son las mas importantes dentro del conjunto analizado.
- Si en una iteracion todos los deltas son iguales, el algoritmo se detiene y conserva las variables restantes, porque no hay una justificacion clara para eliminar solo una.
- Solo cuando el proceso llega a una sola columna restante se interpreta esa variable como la caracteristica final mas relevante.



In [5]:
# Ejecucion del algoritmo de seleccion de caracteristicas por entropia.
# Cambia log_base para elegir la base del logaritmo.

log_base = 10
ranking_result = entropy_feature_ranking(df, base=log_base)

print('Base del logaritmo =', log_base)

for iteration_result in ranking_result['iteraciones']:
    print(f"\nIteracion {iteration_result['iteracion']}")
    display(iteration_result['tabla'])

    if iteration_result['detener_por_deltas_iguales']:
        print(iteration_result['mensaje'])
        print('Columnas restantes:', iteration_result['columnas_restantes'])
        break

    if len(iteration_result['empatadas']) > 1:
        print('Empate en la menor diferencia de entropia:', iteration_result['empatadas'])
        print('Se elimina la primera por el orden actual de columnas.')
    else:
        print('Mejor candidata a eliminar:', iteration_result['empatadas'][0])

    print('Columna eliminada:', iteration_result['columna_eliminada'])
    print('Columnas restantes:', iteration_result['columnas_restantes'])

print('\nRanking final de eliminacion:', ranking_result['ranking_eliminacion'])

if ranking_result['detenido_por_deltas_iguales']:
    print('El proceso se detuvo porque todos los deltas de la ultima iteracion fueron iguales.')
    print('No se selecciona una unica variable final con este criterio en ese punto.')
    print('Columnas restantes:', ranking_result['columnas_restantes'])
else:
    print('Caracteristica final mas relevante:', ranking_result['columna_final'])


Base del logaritmo = 10

Iteracion 1


,iteracion,columna_removida_candidata,entropia_actual,entropia_sin_columna,delta_entropia,empate_minimo
0,1,F1,1.105738,0.00000,1.105738,False
1,1,F2,1.105738,1.20412,-0.098382,True
2,1,F3,1.105738,1.20412,-0.098382,True


Empate en la menor diferencia de entropia: ['F2', 'F3']
Se elimina la primera por el orden actual de columnas.
Columna eliminada: F2
Columnas restantes: ['F1', 'F3']

Iteracion 2


,iteracion,columna_removida_candidata,entropia_actual,entropia_sin_columna,delta_entropia,empate_minimo
0,2,F1,1.20412,0.0,1.20412,True
1,2,F3,1.20412,0.0,1.20412,True


No se elimina ninguna variable porque todos los deltas son iguales y retirar una variable puede eliminar variabilidad del sistema.
Columnas restantes: ['F1', 'F3']

Ranking final de eliminacion: ['F2']
El proceso se detuvo porque todos los deltas de la ultima iteracion fueron iguales.
No se selecciona una unica variable final con este criterio en ese punto.
Columnas restantes: ['F1', 'F3']
